In [2]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")

print("Path to dataset files:", path)

c:\Users\eglha\anaconda3\envs\torch_gen\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\eglha\anaconda3\envs\torch_gen\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 23.6G/23.6G [07:59<00:00, 52.8MB/s]

Extracting files...


Path to dataset files: C:\Users\eglha\.cache\kagglehub\datasets\awsaf49\asvpoof-2019-dataset\versions\1


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammedabdeldayem/avsspoof-2021")

print("Path to dataset files:", path)

100%|██████████| 58.0G/58.0G [21:16<00:00, 48.8MB/s]  

Extracting files...


Path to dataset files: C:\Users\eglha\.cache\kagglehub\datasets\mohammedabdeldayem\avsspoof-2021\versions\7


In [1]:
import os
import random
from huggingface_hub import hf_hub_download
import numpy as np
import pandas as pd
from tqdm import tqdm
import glob

import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from transformers import (
    Wav2Vec2Model
)
import kagglehub
from huggingface_hub import hf_hub_download

from sklearn.metrics import (
    roc_curve,
    confusion_matrix,
    classification_report
)

from scipy.optimize import brentq
from scipy.interpolate import interp1d

import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

torch.backends.cudnn.benchmark = True

# ============================================================
# OUTPUT DIRECTORIES
# ============================================================


RESULTS_DIR = "wav2vec2_4_results"
CACHE_DIR = "wav2vec2_4_cache"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ============================================================
# HYPERPARAMETERS
# ============================================================

SR = 16000
MAX_LEN = 4 * SR
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-5
NUM_WORKERS = 4

# ============================================================
# DATASET PATHS
# ============================================================

BASE_2019 = r"C:\Users\eglha\.cache\kagglehub\datasets\awsaf49\asvpoof-2019-dataset\versions\1\LA\LA"

TRAIN_AUDIO_DIR = os.path.join(
    BASE_2019,
    "ASVspoof2019_LA_train",
    "flac"
)

DEV_AUDIO_DIR = os.path.join(
    BASE_2019,
    "ASVspoof2019_LA_dev",
    "flac"
)

TRAIN_PROTOCOL = os.path.join(
    BASE_2019,
    "ASVspoof2019_LA_cm_protocols",
    "ASVspoof2019.LA.cm.train.trn.txt"
)

DEV_PROTOCOL = os.path.join(
    BASE_2019,
    "ASVspoof2019_LA_cm_protocols",
    "ASVspoof2019.LA.cm.dev.trl.txt"
)

# ============================================================
# 2021 DF
# ============================================================

BASE_2021 = os.path.join(
    "datasets",
    "ASVspoof2021_DF_eval_part00",
    "ASVspoof2021_DF_eval"
)

TEST_AUDIO_DIR = os.path.join(
    BASE_2021,
    "flac"
)



DF_KEY_FILE = os.path.join(
    "keys",
    "DF",
    "CM",
    "trial_metadata.txt"
)

TEST_PROTOCOL = DF_KEY_FILE

# ============================================================
# SEED
# ============================================================

def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

seed_everything()

# ============================================================
# EER
# ============================================================

def compute_eer(y_true, y_score):

    fpr, tpr, _ = roc_curve(
        y_true,
        y_score
    )

    eer = brentq(
        lambda x: 1. - x - interp1d(fpr, tpr)(x),
        0.,
        1.
    )

    return eer * 100

# ============================================================
# CACHE AUDIO
# ============================================================

import soundfile as sf

def process_audio(audio_path):

    try:

        # ============================================
        # TRY SOUNDFile FIRST
        # ============================================

        waveform, sr = sf.read(audio_path)

        # stereo -> mono
        if len(waveform.shape) > 1:

            waveform = waveform.mean(axis=1)

        # resample if needed
        if sr != SR:

            waveform = librosa.resample(
                waveform,
                orig_sr=sr,
                target_sr=SR
            )

    except Exception as e:

        try:

            # ============================================
            # FALLBACK TO LIBROSA
            # ============================================

            waveform, sr = librosa.load(
                audio_path,
                sr=SR,
                mono=True
            )

        except Exception as e2:

            print(f"FAILED TO LOAD: {audio_path}")

            return None

    # ============================================
    # PAD / TRUNCATE
    # ============================================

    if len(waveform) > MAX_LEN:

        waveform = waveform[:MAX_LEN]

    else:

        waveform = np.pad(
            waveform,
            (0, MAX_LEN - len(waveform))
        )

    waveform = torch.tensor(
        waveform
    ).float()

    return waveform
# ============================================================
# PRECACHE AUDIO
# ============================================================

def cache_audio_dataset(audio_dir):

    print(f"\nCaching: {audio_dir}")

    files = [
        f for f in os.listdir(audio_dir)
        if f.endswith(".flac")
    ]

    for file in tqdm(files):

        cache_path = os.path.join(
            CACHE_DIR,
            file.replace(".flac", ".pt")
        )

        if os.path.exists(cache_path):
            continue

        audio_path = os.path.join(
            audio_dir,
            file
        )

        waveform = process_audio(
            audio_path
        )
        
        if waveform is None:
        
            continue
        
        torch.save(
            waveform,
            cache_path
        )
# ============================================================
# DATASET
# ============================================================

class ASVSpoofDataset(Dataset):

    def __init__(
        self,
        protocol_file,
        dataset_type="2019"
    ):

        self.data = []

        with open(protocol_file, "r") as f:

            lines = f.readlines()

        # ============================================
        # 2019
        # ============================================

        if dataset_type == "2019":

            for line in lines:

                parts = line.strip().split()

                file_id = parts[1]

                label = parts[-1]

                label = (
                    1 if label == "bonafide"
                    else 0
                )

                self.data.append(
                    (file_id, label)
                )

        # ============================================
        # 2021
        # ============================================

        
        else:
            for line in lines:
                parts = line.strip().split()
        
                # Expected format:
                # speaker  file_id  codec  source  attack  label  ...
                # Example:
                # LA_0023 DF_E_2000011 nocodec asvspoof A14 spoof notrim ...
        
                if len(parts) < 6:
                    continue
        
                file_id = parts[1]      # DF_E_2000011
                label_str = parts[5]    # spoof / bonafide
        
                if label_str not in ["spoof", "bonafide"]:
                    continue
        
                label = 1 if label_str == "bonafide" else 0
        
                self.data.append((file_id, label))


    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        file_id, label = self.data[idx]

        cache_path = os.path.join(
            CACHE_DIR,
            file_id + ".pt"
        )

        if not os.path.exists(cache_path):
        
            return self.__getitem__(
                random.randint(0, len(self.data)-1)
            )
        
        waveform = torch.load(cache_path)

        return waveform, torch.tensor(label).long()

# ============================================================
# MODEL
# ============================================================

class Wav2Vec2Deepfake(nn.Module):

    def __init__(self):

        super().__init__()

        self.wav2vec = Wav2Vec2Model.from_pretrained(
            "facebook/wav2vec2-base",
            use_safetensors=True
        )

        hidden_size = (
            self.wav2vec.config.hidden_size
        )

        # ============================================
        # FREEZE MOST OF MODEL
        # ============================================

        for param in self.wav2vec.parameters():

            param.requires_grad = False

        # ============================================
        # UNFREEZE LAST 2 ENCODER LAYERS
        # ============================================

        for layer in self.wav2vec.encoder.layers[-2:]:

            for param in layer.parameters():

                param.requires_grad = True

        # ============================================
        # CLASSIFIER
        # ============================================

        self.classifier = nn.Sequential(

            nn.Linear(hidden_size, 256),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256, 2)
        )

    def forward(self, x):

        outputs = self.wav2vec(
            x
        )

        hidden_states = (
            outputs.last_hidden_state
        )

        pooled = hidden_states.mean(
            dim=1
        )

        logits = self.classifier(
            pooled
        )

        return logits

# ============================================================
# TRAIN
# ============================================================
'''
def train_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler
):

    model.train()

    total_loss = 0

    for waveforms, labels in tqdm(loader):

        waveforms = waveforms.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():

            outputs = model(
                waveforms
            )

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)
'''
# ============================================================
# EVALUATE
# ============================================================

def evaluate(model, loader):

    model.eval()

    y_true = []

    y_scores = []

    y_pred = []

    with torch.no_grad():

        for waveforms, labels in tqdm(loader):

            waveforms = waveforms.to(
                DEVICE,
                non_blocking=True
            )

            outputs = model(
                waveforms
            )

            probs = torch.softmax(
                outputs,
                dim=1
            )[:, 1]

            preds = (
                probs >= 0.5
            ).long()

            y_scores.extend(
                probs.cpu().numpy()
            )

            y_pred.extend(
                preds.cpu().numpy()
            )

            y_true.extend(
                labels.numpy()
            )

    eer = compute_eer(
        y_true,
        y_scores
    )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    report = classification_report(
        y_true,
        y_pred,
        digits=4
    )

    return {

        "eer": eer,

        "y_true": y_true,

        "y_scores": y_scores,

        "y_pred": y_pred,

        "confusion_matrix": cm,

        "report": report
    }


# ============================================================
# FIND LATEST CHECKPOINT
# ============================================================
'''
def get_latest_checkpoint():

    checkpoint_files = glob.glob(
        os.path.join(
            CHECKPOINT_DIR,
            "epoch_*.pth"
        )
    )

    if len(checkpoint_files) == 0:
        return None

    checkpoint_files.sort(
        key=lambda x: int(
            x.split("epoch_")[1].split(".pth")[0]
        )
    )

    latest_checkpoint = checkpoint_files[-1]

    return latest_checkpoint
'''   


c:\Users\eglha\anaconda3\envs\torch_gen\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\eglha\anaconda3\envs\torch_gen\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Using device: cuda


'\ndef get_latest_checkpoint():\n\n    checkpoint_files = glob.glob(\n        os.path.join(\n            CHECKPOINT_DIR,\n            "epoch_*.pth"\n        )\n    )\n\n    if len(checkpoint_files) == 0:\n        return None\n\n    checkpoint_files.sort(\n        key=lambda x: int(\n            x.split("epoch_")[1].split(".pth")[0]\n        )\n    )\n\n    latest_checkpoint = checkpoint_files[-1]\n\n    return latest_checkpoint\n'

In [2]:
 
# ============================================================
# MAIN
# ============================================================

def main():

    wav2vec_name='best_model.pth'
    latest_checkpoint=hf_hub_download(
        repo_id='rde6mn/no_aug_w2v_4s',
        filename=wav2vec_name)
    final_checkpoint = latest_checkpoint
    # ============================================
    # CACHE AUDIO
    # ============================================

    cache_audio_dataset(
        TRAIN_AUDIO_DIR
    )

    #cache_audio_dataset(DEV_AUDIO_DIR)

    #cache_audio_dataset(TEST_AUDIO_DIR)

    # ============================================
    # DATASETS
    # ============================================

    # ============================================
    # FULL 2019 DATASET
    # ============================================
    
    full_dataset = ASVSpoofDataset(
        TRAIN_PROTOCOL,
        dataset_type="2019"
    )
    
    # ============================================
    # TRAIN / VAL / TEST SPLIT
    # ============================================
    
    train_size = int(0.8 * len(full_dataset))
    
    val_size = int(0.1 * len(full_dataset))
    
    test_size = (
        len(full_dataset)
        - train_size
        - val_size
    )
    
    train_dataset, val_dataset, test_dataset = random_split(
        full_dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )


    
    # ============================================
    # AUGMENTATION SETTINGS
    # ============================================
    
    train_dataset.dataset.training = False
    

    
    # ============================================
    # DATALOADERS
    # ============================================
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    # ============================================
    # MODEL
    # ============================================

    model = Wav2Vec2Deepfake().to(
        DEVICE
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(

        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),

        lr=LR
    )
    model.load_state_dict(torch.load(final_checkpoint, map_location=DEVICE)["model_state_dict"])

    
    # ============================================
    # RESUME FROM CHECKPOINT
    # ============================================
    
    # ============================================================
    # CHECKPOINT HANDLING (FIXED)
    # ============================================================
    '''
    latest_checkpoint = get_latest_checkpoint()
    
    if latest_checkpoint is None:
        # No checkpoints at all
        print("\nNo checkpoint found. Starting fresh training.")
        last_epoch = 0
        skip_training = False
    
    else:
        # Extract last completed epoch number
        last_epoch = int(latest_checkpoint.split("epoch_")[1].split(".pth")[0])
        print(f"\nFound checkpoint: {latest_checkpoint} (epoch {last_epoch})")
    
        # Decide whether to skip training
        if last_epoch >= EPOCHS:
            print("Training already completed. Skipping directly to testing.")
            skip_training = True
        else:
            print(f"Resuming training from epoch {last_epoch + 1}.")
            skip_training = False
    
    # ============================================================
    # LOAD CHECKPOINT IF RESUMING
    # ============================================================
    
    start_epoch = last_epoch
    
    if not skip_training and latest_checkpoint is not None:
        checkpoint = torch.load(latest_checkpoint, map_location=DEVICE)
    
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    
        train_losses = checkpoint.get("train_losses", [])
        val_eers = checkpoint.get("val_eers", [])
        best_val_eer = checkpoint.get("best_val_eer", 999)
    
    else:
        # Fresh start or skipping training
        train_losses = []
        val_eers = []
        best_val_eer = 999



    # ============================================
    # TRAINING
    # ============================================

    if not skip_training:

        for epoch in range(start_epoch, EPOCHS):
    
            print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
            train_loss = train_epoch(
                model,
                train_loader,
                optimizer,
                criterion,
                scaler
            )
    
            results = evaluate(
                model,
                val_loader
            )
    
            val_eer = results["eer"]
    
            train_losses.append(train_loss)
            val_eers.append(val_eer)
    
            print(
                f"Train Loss: {train_loss:.4f}"
            )
    
            print(
                f"Val EER: {val_eer:.2f}%"
            )
    
            checkpoint = {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "train_loss":
                    train_loss,
                "val_eer":
                    val_eer,
                "train_losses":
                    train_losses,
                "val_eers":
                    val_eers
            }
    
            torch.save(
                checkpoint,
                os.path.join(
                    CHECKPOINT_DIR,
                    f"epoch_{epoch+1}.pth"
                )
            )
    
            if val_eer < best_val_eer:
    
                best_val_eer = val_eer
    
                torch.save(
                    checkpoint,
                    os.path.join(
                        CHECKPOINT_DIR,
                        "best_model.pth"
                    )
                )
    
                print("Saved best model.")

        # ========================================
        # SAVE BEST MODEL
        # ========================================

        if val_eer < best_val_eer:

            best_val_eer = val_eer

            torch.save(

                checkpoint,

                os.path.join(
                    CHECKPOINT_DIR,
                    "best_model.pth"
                )
            )

            print(
                "Saved best model."
            )

    # ============================================
    # LOAD BEST MODEL
    # ============================================

    if skip_training:
        # Load final epoch checkpoint
        checkpoint = torch.load(
            os.path.join(CHECKPOINT_DIR, f"epoch_{EPOCHS}.pth"),
            map_location=DEVICE
        )
    else:
        # Load best model after training
        checkpoint = torch.load(
            os.path.join(CHECKPOINT_DIR, "best_model.pth"),
            map_location=DEVICE
        )
    
    model.load_state_dict(checkpoint["model_state_dict"])
'''

    # ============================================
    # FINAL TEST
    # ============================================
'''
    results = evaluate(
        model,
        test_loader
    )

    print(
        f"\nASVspoof2021 DF EER: "
        f"{results['eer']:.2f}%"
    )

    # ============================================
    # CONFUSION MATRIX
    # ============================================

    tn, fp, fn, tp = results[
        "confusion_matrix"
    ].ravel()

    print(f"TP: {tp}")

    print(f"TN: {tn}")

    print(f"FP: {fp}")

    print(f"FN: {fn}")

    # ============================================
    # SAVE REPORT
    # ============================================

    with open(

        os.path.join(
            RESULTS_DIR,
            "classification_report.txt"
        ),

        "w"
    ) as f:

        f.write(
            results["report"]
        )

    # ============================================
    # SAVE PREDICTIONS
    # ============================================

    df = pd.DataFrame({

        "true_label":
            results["y_true"],

        "prediction":
            results["y_pred"],

        "score":
            results["y_scores"]
    })

    df.to_csv(

        os.path.join(
            RESULTS_DIR,
            "test_predictions.csv"
        ),

        index=False
    )

    # ============================================
    # SAVE METRICS
    # ============================================

    metrics_df = pd.DataFrame({

        "epoch":
            list(range(
                1,
                EPOCHS + 1
            )),

        "train_loss":
            train_losses,

        "val_eer":
            val_eers
    })

    metrics_df.to_csv(

        os.path.join(
            RESULTS_DIR,
            "training_metrics.csv"
        ),

        index=False
    )

    # ============================================
    # TRAIN LOSS CURVE
    # ============================================

    plt.figure(figsize=(8,5))

    plt.plot(train_losses)

    plt.xlabel("Epoch")

    plt.ylabel("Train Loss")

    plt.title("Training Loss Curve")

    plt.savefig(

        os.path.join(
            RESULTS_DIR,
            "train_loss_curve.png"
        )
    )

    plt.close()

    print("\nDone.")
'''
# ============================================================

if __name__ == "__main__":

    main()


Caching: C:\Users\eglha\.cache\kagglehub\datasets\awsaf49\asvpoof-2019-dataset\versions\1\LA\LA\ASVspoof2019_LA_train\flac


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 15294.98it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\eglha\anaconda3\envs\torch_gen\lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5070 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 s

In [3]:

from huggingface_hub import hf_hub_download

model = Wav2Vec2Deepfake().to(DEVICE)

wav2vec_name='best_model.pth'
latest_checkpoint=hf_hub_download(
    repo_id='rde6mn/no_aug_w2v_4s',
    filename=wav2vec_name
)
checkpoint = torch.load(latest_checkpoint, map_location=DEVICE)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad,model.parameters()),lr=LR)   
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])




Loading weights: 100%|██████████| 211/211 [00:00<00:00, 19994.54it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\eglha\AppData\Local\Temp\ipykernel_14468\3016784762.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during un

In [4]:
class ASVSpoofDataset(Dataset):
    def __init__(
        self,
        protocol_file,
        dataset_type="2019"
    ):
        self.data = []
        with open(protocol_file, "r") as f:
            lines = f.readlines()
        # ============================================
        # 2019
        # ============================================
        if dataset_type == "2019":
            for line in lines:
                parts = line.strip().split()
                file_id = parts[1]
                label = parts[-1]
                label = (
                    1 if label == "bonafide"
                    else 0
                )
                self.data.append(
                    (file_id, label)
                )
        # ============================================
        # 2021
        # ============================================
        else:
            for line in lines:
                parts = line.strip().split()
                # Expected format:
                # speaker  file_id  codec  source  attack  label  ...
                # Example:
                # LA_0023 DF_E_2000011 nocodec asvspoof A14 spoof notrim ...
                if len(parts) < 6:
                    continue
                file_id = parts[1]      # DF_E_2000011
                label_str = parts[5]    # spoof / bonafide
                if label_str not in ["spoof", "bonafide"]:
                    continue
                label = 1 if label_str == "bonafide" else 0
                self.data.append((file_id, label))
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        file_id, label = self.data[idx]
        cache_path = os.path.join(
            CACHE_DIR,
            file_id + ".pt"
        )
        if not os.path.exists(cache_path):
            return self.__getitem__(
                random.randint(0, len(self.data)-1)
            )
        waveform = torch.load(cache_path)
        return waveform, torch.tensor(label).long()

In [5]:
full_dataset = ASVSpoofDataset(
    TRAIN_PROTOCOL,
    dataset_type="2019"
)

In [6]:
full_dataset[1]

C:\Users\eglha\AppData\Local\Temp\ipykernel_14468\2524288094.py:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  waveform = torch.load(cache_path)


(tensor([-0.0021, -0.0020, -0.0019,  ..., -0.0096, -0.0270, -0.0323]),
 tensor(1))

In [16]:
def make_spectrogram(dataset_item, sr=20):
    waveform, label = dataset_item
    # Compute the spectrogram
    S = librosa.stft(waveform.numpy())
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    # Plot the spectrogram
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Spectrogram for label: {"bonafide" if label.item() == 0 else "spoof"}')
    plt.tight_layout()
    plt.show()

In [7]:
random_item=full_dataset[random.randint(0, len(full_dataset)-1)]
make_spectrogram(random_item)

C:\Users\eglha\AppData\Local\Temp\ipykernel_14468\2524288094.py:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  waveform = torch.load(cache_path)


NameError: name 'make_spectrogram' is not defined

In [8]:
data_loader = DataLoader(
        full_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

In [ ]:
DEVICE='cpu'
def predict(model, dataloader, pred_limit=None):
    with torch.no_grad():
        model.eval()
        predictions = []
        i=0
        for waveform, label in tqdm(dataloader, desc="Predicting"):

            waveform = waveform.to(DEVICE)  # Add batch dimension
            outputs = model(waveform)
            probs = torch.softmax(outputs, dim=1)[:, 1]  # Get probability of class 1 (bonafide)
            predictions.append((probs.item(), label.item()))
            i+=1
            if pred_limit is not None and i >= pred_limit:
                break
    return predictions
    

predictions = predict(model, data_loader, pred_limit=50)

Predicting:   0%|          | 0/1587 [00:00<?, ?it/s]